# HELM — Inference & Model Internals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/HELM/blob/master/notebooks/demo_inference.ipynb)

Explore what HELM receives as input, how the hierarchy is structured, what the model outputs, and how predictions map to leaf and ancestor labels.

## 1. Setup

In [ ]:
%cd /content
!rm -rf HELM
!git clone https://github.com/marjanstoimchev/HELM.git
%cd HELM
!pip install -q -r requirements.txt

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || echo "PyG extensions installed via fallback"
print("Setup complete!")

In [ ]:
import os, sys, gc, glob, yaml
import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning import seed_everything
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
from matplotlib.patches import Patch, FancyBboxPatch
import networkx as nx

sys.path.insert(0, ".")

from data.dataset_pipeline import DatasetPipeline
from data.hierarchy import create_edge_index, extract_paths, process_hierarchy_config, extend_ys, build_hierarchy_mapping
from datamodules.base_datamodule import BaseDataModule
from models.model import h_deit_base_embedding
from augmentations import Preprocess
from callbacks import ModelCheckpoint_, EarlyStopping_, RichProgressBar_
from utils.utils import Dotdict, calculate_metrics, predict
from trainers import get_trainer_class

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.titleweight": "bold",
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load hierarchy and dataset

In [ ]:
DATASET = "ucm"
DATASET_CONFIG = f"configs/dataset/{DATASET}.yaml"
SEED = 42

with open(DATASET_CONFIG) as f:
    ds_cfg = yaml.safe_load(f)

# Parse hierarchy
h_out = process_hierarchy_config(DATASET_CONFIG)
leaf_labels = ds_cfg["leaf_labels"]
leaf_paths = h_out.leaf_paths.to_dict()
intermediate_paths = h_out.intermediate_paths.to_dict()

print(f"Dataset: {ds_cfg['name']}")
print(f"Leaf labels (M_l): {h_out.num_classes}")
print(f"Intermediate nodes (M_h): {len(intermediate_paths)}")
print(f"Total nodes (M = M_l + M_h): {h_out.num_classes_extended}")

## 3. Hierarchy structure

The label hierarchy is a tree. Leaf nodes are the original dataset labels. Intermediate nodes are parent categories added by HELM.

In [ ]:
# ── Tree drawing utilities ───────────────────────────────────────────────

C_LEAF = '#2E86AB'     # leaf node
C_INTER = '#E8871E'    # intermediate node
C_ROOT = '#2c3e50'     # root
C_EDGE = '#7f8c9a'     # edges
C_TP = '#27ae60'       # true positive
C_FN = '#e74c3c'       # false negative (missed)
C_FP = '#e67e22'       # false positive
C_ANC = '#eaf0f6'      # ancestor (neutral)

NODE_STYLES = {
    'leaf':  dict(fc=C_LEAF,  ec='#1a6d8e', tc='white',   fw='bold',   lw=1.6, alpha=1.0),
    'inter': dict(fc='#f0f4f8', ec='#a0b0c0', tc='#3d5166', fw='normal', lw=1.2, alpha=0.92),
    'tp':    dict(fc=C_TP,    ec='#1a7a44', tc='white',   fw='bold',   lw=1.8, alpha=1.0),
    'fn':    dict(fc=C_FN,    ec='#a93226', tc='white',   fw='bold',   lw=1.8, alpha=1.0),
    'fp':    dict(fc=C_FP,    ec='#b35a0e', tc='white',   fw='bold',   lw=1.8, alpha=1.0),
    'anc':   dict(fc=C_ANC,   ec='#a0b0c0', tc='#3d5166', fw='normal', lw=1.2, alpha=0.92),
}

ROOT_SENTINEL = '__root__'


def build_nx_tree(hierarchy, parent=None):
    """Convert YAML hierarchy dict to a networkx DiGraph."""
    G = nx.DiGraph()
    for key, value in hierarchy.items():
        G.add_node(key, is_leaf=(not value))
        if parent is not None:
            G.add_edge(parent, key)
        if isinstance(value, dict) and value:
            sub = build_nx_tree(value, parent=key)
            G = nx.compose(G, sub)
    return G


def count_leaves_nx(G, node):
    children = list(G.successors(node))
    if not children:
        return 1
    return sum(count_leaves_nx(G, c) for c in children)


def hierarchy_layout(G, root, width=3.5, vert_gap=0.32, vert_loc=0, xcenter=0.5):
    """Compute tree positions with leaf-proportional horizontal spacing."""
    pos = {}

    def _layout(node, w, vg, vloc, xc):
        pos[node] = (xc, vloc)
        children = list(G.successors(node))
        if not children:
            return
        lc = [count_leaves_nx(G, c) for c in children]
        total = sum(lc)
        left = xc - w / 2
        for child, n_leaves in zip(children, lc):
            child_w = w * n_leaves / total
            _layout(child, child_w, vg, vloc - vg, left + child_w / 2)
            left += child_w

    _layout(root, width, vert_gap, vert_loc, xcenter)
    return pos


def shorten(name, mx=16):
    name = name.replace('_', ' ').replace('-', ' ')
    if len(name) <= mx:
        return name
    words = name.split()
    if len(words) >= 3:
        return words[0][:8] + '.\n' + ' '.join(words[-2:])[:mx]
    if len(words) == 2:
        return words[0][:8] + '.\n' + words[1][:8]
    return name[:mx - 1] + '.'


def draw_orthogonal_edge(ax, x0, y0, x1, y1, color=C_EDGE, lw=1.0):
    mid_y = (y0 + y1) / 2
    ax.plot([x0, x0], [y0, mid_y], color=color, lw=lw, solid_capstyle='round', zorder=2)
    ax.plot([x0, x1], [mid_y, mid_y], color=color, lw=lw, solid_capstyle='round', zorder=2)
    ax.plot([x1, x1], [mid_y, y1], color=color, lw=lw, solid_capstyle='round', zorder=2)


def draw_node_box(ax, x, y, text, style, fontsize=9):
    pad = 0.45 if len(text) > 10 else 0.38
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, fontfamily='serif',
            fontweight=style['fw'], color=style['tc'],
            bbox=dict(boxstyle=f'round,pad={pad}',
                      fc=style['fc'], ec=style['ec'],
                      lw=style['lw'], alpha=style['alpha']),
            zorder=4)


def draw_full_hierarchy(ax, hierarchy, title=None):
    """Render the complete hierarchy tree on an axes."""
    G = build_nx_tree(hierarchy)
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]

    G.add_node(ROOT_SENTINEL)
    for r in roots:
        G.add_edge(ROOT_SENTINEL, r)

    pos = hierarchy_layout(G, ROOT_SENTINEL, width=4.0, vert_gap=0.28)

    for u, v in G.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        draw_orthogonal_edge(ax, x0, y0 - 0.03, x1, y1 + 0.03)

    for n in G.nodes():
        if n == ROOT_SENTINEL:
            x, y = pos[n]
            ax.plot(x, y, 'o', ms=8, mfc='white', mec=C_ROOT, mew=2.0, zorder=5)
            continue
        x, y = pos[n]
        is_leaf = G.out_degree(n) == 0
        style = NODE_STYLES['leaf'] if is_leaf else NODE_STYLES['inter']
        draw_node_box(ax, x, y, shorten(n), style, fontsize=8.5 if is_leaf else 7.5)

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.3, max(xs) + 0.3)
    ax.set_ylim(min(ys) - 0.15, max(ys) + 0.12)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, pad=10)


def draw_prediction_tree(ax, hierarchy, all_names, true_set, pred_set, title=None, bg='white'):
    """Draw hierarchy tree with TP/FN/FP coloring for a single prediction."""
    ax.set_facecolor(bg)

    # Find relevant nodes (ancestors of any active leaf)
    def get_ancestors(node, h, path=None):
        if path is None: path = []
        for key, value in h.items():
            if key == node: return path + [key]
            if isinstance(value, dict) and value:
                r = get_ancestors(node, value, path + [key])
                if r: return r
        return None

    relevant = set()
    for idx in true_set | pred_set:
        if idx < len(all_names):
            anc = get_ancestors(all_names[idx], hierarchy)
            if anc: relevant.update(anc)
    if not relevant:
        ax.axis('off')
        return

    # Build subgraph of relevant nodes
    def build_sub(h, rel, parent=None):
        G = nx.DiGraph()
        for key, value in h.items():
            if key in rel:
                G.add_node(key)
                if parent and parent in rel:
                    G.add_edge(parent, key)
                if isinstance(value, dict) and value:
                    G = nx.compose(G, build_sub(value, rel, key))
        return G

    G = build_sub(hierarchy, relevant)
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]
    G.add_node(ROOT_SENTINEL)
    for r in roots:
        G.add_edge(ROOT_SENTINEL, r)

    pos = hierarchy_layout(G, ROOT_SENTINEL, width=3.2, vert_gap=0.30)

    for u, v in G.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        draw_orthogonal_edge(ax, x0, y0 - 0.03, x1, y1 + 0.03)

    for n in G.nodes():
        if n == ROOT_SENTINEL:
            x, y = pos[n]
            ax.plot(x, y, 'o', ms=7, mfc='white', mec=C_EDGE, mew=1.8, zorder=5)
            continue
        x, y = pos[n]
        idx = all_names.index(n) if n in all_names else -1
        is_tp = idx in true_set and idx in pred_set
        is_fn = idx in true_set and idx not in pred_set
        is_fp = idx in pred_set and idx not in true_set

        if is_tp:   style = NODE_STYLES['tp']
        elif is_fn: style = NODE_STYLES['fn']
        elif is_fp: style = NODE_STYLES['fp']
        else:       style = NODE_STYLES['anc']

        draw_node_box(ax, x, y, shorten(n), style, fontsize=9 if G.out_degree(n) == 0 else 7.5)

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.3, max(xs) + 0.3)
    ax.set_ylim(min(ys) - 0.15, max(ys) + 0.12)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=11, pad=8)


# ── Render full UCM hierarchy ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 7))
draw_full_hierarchy(ax, ds_cfg["hierarchy"],
                    title=f"UCM Label Hierarchy ({h_out.num_classes} leaves, {len(intermediate_paths)} intermediate nodes)")

fig.legend(
    [Patch(**{k: NODE_STYLES['leaf'][k] for k in ['fc','ec']}),
     Patch(**{k: NODE_STYLES['inter'][k] for k in ['fc','ec']})],
    [f"Leaf label ({h_out.num_classes})", f"Intermediate node ({len(intermediate_paths)})"],
    loc="lower center", ncol=2, fontsize=11, frameon=False,
)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 4. Leaf vs ancestor labels

In HMLC mode, when a leaf label is active, all its ancestors are activated too. This shows the mapping.

In [ ]:
# Build ordered leaf paths + intermediate paths
ordered_lp = {k: leaf_paths[k] for k in leaf_labels}
final_strings = {**ordered_lp, **intermediate_paths}
all_label_names = list(final_strings.keys())

# Reverse lookup: path code -> label name
code_to_name = {v: k for k, v in final_strings.items()}

# Show ancestor chain for each leaf
print(f"{'Leaf Label':<25s} {'Ancestors (root \u2192 leaf)'}")
print("\u2500" * 70)
hierarchy_map = build_hierarchy_mapping(final_strings)
for leaf in leaf_labels[:5]:  # show first 5
    ancestors_codes = hierarchy_map[leaf]
    chain = [code_to_name.get(c, c) for c in ancestors_codes if c in code_to_name]
    print(f"{leaf:<25s} {' \u2192 '.join(chain)}")
print(f"... ({len(leaf_labels)} total leaves)")

## 5. What the model sees

HELM receives images as 224x224 tensors. In HMLC mode, the label vector is extended from M_l leaf labels to M total labels (leaf + ancestors).

In [ ]:
from lightning import seed_everything
seed_everything(SEED, workers=True)

pipeline = DatasetPipeline(yaml_path=DATASET_CONFIG, seed=SEED, cache_dir="./Datasets/mlc_datasets_npy")
outputs = pipeline.run_pipeline(fraction_labeled=None)

num_leaves = pipeline.num_classes
num_classes = pipeline.num_classes_extended

print("\u2500\u2500 Model Input \u2500\u2500")
print(f"Image shape:         {outputs['X_te'][0].shape}  (C, H, W)")
print(f"Image dtype:         {outputs['X_te'][0].dtype}")
print(f"Image value range:   [{outputs['X_te'][0].min()}, {outputs['X_te'][0].max()}]")

print(f"\n\u2500\u2500 Label Vectors \u2500\u2500")
print(f"Leaf labels (Y_te):  shape {outputs['Y_te'].shape}  ({num_leaves} classes)")
print(f"Full labels (Y_te_h): shape {outputs['Y_te_h'].shape}  ({num_classes} classes = {num_leaves} leaf + {num_classes - num_leaves} ancestors)")

# Show an example
idx = 0
leaf_vec = outputs['Y_te'][idx]
full_vec = outputs['Y_te_h'][idx]
active_leaves = [leaf_labels[i] for i in np.where(leaf_vec > 0)[0]]
active_all = [all_label_names[i] for i in np.where(full_vec > 0)[0]]
active_ancestors = [n for n in active_all if n not in active_leaves]

print(f"\n\u2500\u2500 Example (sample {idx}) \u2500\u2500")
print(f"Active leaf labels:     {active_leaves}")
print(f"Activated ancestors:    {active_ancestors}")
print(f"Full extended vector:   {active_all}")

## 6. Visualize input samples with hierarchical labels

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "axes.titleweight": "bold",
})

N_SHOW = 4
indices = np.random.choice(len(outputs['X_te']), N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW, figsize=(4.5 * N_SHOW, 8),
                          gridspec_kw={"height_ratios": [1, 1.3]})

for col, idx in enumerate(indices):
    # Image
    img = outputs['X_te'][idx]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    # Labels
    leaf_vec = outputs['Y_te'][idx]
    full_vec = outputs['Y_te_h'][idx]
    active_leaves = [leaf_labels[i] for i in np.where(leaf_vec > 0)[0]]
    active_all = [all_label_names[i] for i in np.where(full_vec > 0)[0]]
    active_ancestors = [n for n in active_all if n not in active_leaves]

    axes[0, col].set_title(", ".join(active_leaves) or "none", fontsize=9)

    # Bar chart: full extended label vector
    ax = axes[1, col]
    active_idx = np.where(full_vec > 0)[0]
    names = [all_label_names[i] for i in active_idx]
    colors = ["#2E86AB" if all_label_names[i] in active_leaves else "#E8871E" for i in active_idx]

    if len(names) > 0:
        ax.barh(range(len(names)), [1]*len(names), color=colors, edgecolor="white")
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.5)
        ax.set_xticks([])
    if col == 0:
        ax.set_ylabel("Active labels", fontsize=10)

# Legend
from matplotlib.patches import Patch
fig.legend(
    [Patch(color="#2E86AB"), Patch(color="#E8871E")],
    ["Leaf label", "Ancestor (activated by hierarchy)"],
    loc="lower center", ncol=2, fontsize=10, frameon=False,
)
fig.suptitle(f"UCM \u2014 Images with Hierarchical Labels", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 7. Load pre-trained HELM model

Downloads the pre-trained HELM checkpoint from Google Drive (UCM, `hmlc-sl-graph-byol`, 100% labeled) and runs inference.

In [ ]:
from models.model import h_deit_base_embedding
from trainers import get_trainer_class

METHOD = "hmlc-sl-graph-byol"
GDRIVE_FILE_ID = "106kboTGXisVJ6RWHmpGeBfkWyTE3vHqr"

with open(f"configs/method/{METHOD}.yaml") as f:
    mcfg = yaml.safe_load(f)

edge_index = create_edge_index(hierarchy=pipeline.label_to_predecessors)

config = Dotdict({
    "training": {"lr": 1e-4, "head_lr": 1e-4, "max_lr": 3e-4,
                 "apply_scheduler": False, "epochs": 1, "min_epochs": 1,
                 "patience": 5, "lr_schedule_patience": 5,
                 "accumulate_grad_batches": 1, "deterministic": True, "log_every_n_steps": 1},
    "dataset": {"name": DATASET, "folder_name": DATASET, "num_classes": num_leaves},
})

backbone = h_deit_base_embedding(num_classes=num_classes, pretrained=False)
trainer_cls = get_trainer_class(mcfg["training_mode"], mcfg["learning_task"], mcfg["apply_edges"], mcfg["apply_byol"])
model = trainer_cls(config, backbone, num_leaves, mcfg["learning_task"], edge_index)

# Download pre-trained checkpoint from Google Drive
model_dir = f"saved_models/{DATASET}/{METHOD}/fraction_100/seed_{SEED}"
model_path = os.path.join(model_dir, "model.ckpt")

if not os.path.exists(model_path):
    !pip install -q gdown
    import gdown
    os.makedirs(model_dir, exist_ok=True)
    print("Downloading pre-trained HELM checkpoint...")
    gdown.download(id=GDRIVE_FILE_ID, output=model_path, quiet=False)

# Load checkpoint — map to current device
map_loc = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
ckpt = torch.load(model_path, map_location=map_loc, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"Loaded pre-trained HELM checkpoint from {model_path}")

# Show what the model outputs for a single sample
from augmentations import Preprocess
preprocess = Preprocess()
sample_img = outputs['X_te'][0]
x = preprocess(np.moveaxis(sample_img, 0, -1)).unsqueeze(0)
y_dummy = torch.zeros(1, num_classes)

with torch.no_grad():
    out = model._forward_eval(x, y_dummy) if hasattr(model, '_forward_eval') else model.forward(x, y_dummy)

logits = out['logits']
probs = torch.sigmoid(logits).squeeze().numpy()

print(f"\n── Model Output ──")
print(f"Logits shape: {logits.shape}  ({num_leaves} leaf classes)")
print(f"\nPredicted probabilities:")
for i, name in enumerate(leaf_labels):
    bar = "█" * int(probs[i] * 30)
    marker = " ◄" if probs[i] > 0.5 else ""
    print(f"  {name:<20s} {probs[i]:.3f} {bar}{marker}")

## 8. Prediction visualization

Top-5 predictions per test sample. Blue = correct leaf label, gray = incorrect.

In [ ]:
from datamodules.base_datamodule import BaseDataModule

transforms = Preprocess()
datamodule = BaseDataModule(outputs, batch_size=16, num_workers=2, transforms=transforms)

accel = "gpu" if torch.cuda.is_available() else "cpu"
inference_trainer = L.Trainer(
    accelerator=accel,
    devices=1 if accel == "cpu" else [0],
    strategy="auto",
    enable_model_summary=False,
)
Y = predict(inference_trainer, model, datamodule)

N_SHOW = 4
n_test = Y["y_true"].shape[0]
indices = np.random.choice(n_test, N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW, figsize=(4 * N_SHOW, 7),
                          gridspec_kw={"height_ratios": [1, 1.2]})

for col, idx in enumerate(indices):
    img = outputs["X_te"][idx]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    true_idx = np.where(Y["y_true"][idx] > 0)[0]
    true_names = [leaf_labels[i] for i in true_idx if i < len(leaf_labels)]
    axes[0, col].set_title(", ".join(true_names) or "none", fontsize=9, wrap=True)

    ax = axes[1, col]
    scores = Y["y_scores"][idx][:len(leaf_labels)]
    top_k = np.argsort(scores)[-5:][::-1]
    top_names = [leaf_labels[k] for k in top_k]
    top_vals = [scores[k] for k in top_k]
    bar_colors = ["#2E86AB" if Y["y_true"][idx][k] > 0 else "#ccc" for k in top_k]

    ax.barh(range(len(top_k)), top_vals, color=bar_colors)
    ax.set_yticks(range(len(top_k)))
    ax.set_yticklabels(top_names, fontsize=9)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()
    if col == 0:
        ax.set_ylabel("Top-5 predictions", fontsize=10)

plt.suptitle("HELM predictions — Blue = correct, Gray = incorrect", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 9. Qualitative hierarchy predictions

For each test sample, the hierarchy tree is drawn with nodes colored by prediction outcome:
- **Green** = correct prediction (TP)
- **Red** = missed label (FN)
- **Orange** = false alarm (FP)
- **Gray** = ancestor node (activated by hierarchy, not directly predicted)

Re-run this cell for different random samples.

In [ ]:
N_EXAMPLES = 4
ROW_BG = ['#ffffff', '#f5f7f9']

indices = np.random.choice(len(Y["y_true"]), N_EXAMPLES, replace=False)

fig = plt.figure(figsize=(22, 5.5 * N_EXAMPLES), dpi=150)
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(N_EXAMPLES, 2, figure=fig,
                          width_ratios=[0.3, 1.0],
                          hspace=0.08, wspace=0.05,
                          left=0.02, right=0.98, top=0.96, bottom=0.06)

for row, si in enumerate(indices):
    bg = ROW_BG[row % 2]

    # Ground truth and predictions (leaf-level)
    true_leaves = set(np.where(Y["y_true"][si] > 0)[0])
    pred_leaves = set(np.where(Y["y_pred"][si] > 0)[0])

    # Map to extended label names (leaves only for TP/FN/FP, ancestors handled inside)
    true_set = {i for i in true_leaves if i < len(all_label_names)}
    pred_set = {i for i in pred_leaves if i < len(all_label_names)}

    # ── Image column ──
    ax_img = fig.add_subplot(outer[row, 0])
    ax_img.set_facecolor(bg)
    img = outputs["X_te"][si]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    ax_img.imshow(np.clip(img, 0, 1), interpolation='lanczos')
    ax_img.set_xticks([]); ax_img.set_yticks([])
    for sp in ax_img.spines.values():
        sp.set_visible(True); sp.set_lw(2.0); sp.set_edgecolor('#2c3e50')

    true_names = [leaf_labels[i] for i in sorted(true_leaves) if i < len(leaf_labels)]
    ax_img.set_title(
        ", ".join(true_names) if true_names else "none",
        fontsize=10, pad=8, fontfamily='serif', color='white', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='#2c3e50', ec='white', lw=1.4, alpha=0.93),
    )

    # ── Tree column ──
    ax_tree = fig.add_subplot(outer[row, 1])
    draw_prediction_tree(ax_tree, ds_cfg["hierarchy"], all_label_names,
                         true_set, pred_set, bg=bg)
    for sp in ax_tree.spines.values():
        sp.set_visible(True); sp.set_lw(0.5); sp.set_edgecolor('#ced4da')

# ── Legend ──
legend_elements = [
    Patch(fc=NODE_STYLES['tp']['fc'], ec=NODE_STYLES['tp']['ec'], label='Correct (TP)', lw=1.5),
    Patch(fc=NODE_STYLES['fn']['fc'], ec=NODE_STYLES['fn']['ec'], label='Missed (FN)', lw=1.5),
    Patch(fc=NODE_STYLES['fp']['fc'], ec=NODE_STYLES['fp']['ec'], label='False positive (FP)', lw=1.5),
    Patch(fc=NODE_STYLES['anc']['fc'], ec=NODE_STYLES['anc']['ec'], label='Ancestor node', lw=1.5),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=12, frameon=False, bbox_to_anchor=(0.5, 0.01),
           prop={'family': 'serif'})

fig.suptitle("HELM Predictions on Hierarchy — UCM Test Set",
             fontsize=15, fontweight='bold', fontfamily='serif', y=0.98)
plt.show()